# 半监督 GNN 空间转录组细胞类型注释
## 转录本级别流水线（不依赖细胞分割）

**数据集**：Ovarian Cancer Flex scRNA-seq + Xenium V1 FFPE

**与肾脏版本的核心差异**
| 步骤 | 肾脏版 | 卵巢版 |
|---|---|---|
| Cell 2 scRNA 加载 | rpy2 + Seurat .rds | **scanpy 直接加载 .h5 + CSV** |
| R 预处理 | 必须先跑 | **可选**（仅用于 UMAP 可视化） |
| 细胞类型注释 | 手动 | **Flex CSV 预标注** |

**缓存文件说明**
| 文件 | 内容 | 节省时间 |
|---|---|---|
| `cache/export/flex_expr.npy` | scRNA 表达矩阵 | 3-5 分钟 |
| `cache/graph/graph_*.pt` | 联合图结构 | 30-60 分钟 |
| `results/models/{name}_best.pt` | 训练权重 | 数小时 |
| `results/predictions/spot_probas_all.pkl` | 节点预测概率 | 5-10 分钟 |

In [8]:
# ============================================================
# Cell 1: 环境配置
# ============================================================
import os, sys, warnings, pickle, json
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ── 路径配置（卵巢癌数据集）─────────────────────────────────
PATHS = {
    'raw': {
        # Xenium 转录本（优先使用 parquet，读取更快）
        'transcripts': 'data/raw/xenium/Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/transcripts.parquet',
        'cells_csv'  : 'data/raw/xenium/Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/cells.csv.gz',
        # Flex scRNA-seq
        'flex_h5'    : 'data/raw/flex/17k_Ovarian_Cancer_scFFPE_count_filtered_feature_bc_matrix.h5',
        'flex_annot' : 'data/raw/flex/FLEX_Ovarian_Barcode_Cluster_Annotation.csv',
    },
    'cache': {
        'export': 'data/cache/export/',
        'graph' : 'data/cache/graph/',
    },
    'output': {
        'models'     : 'results/models/',
        'predictions': 'results/predictions/',
        'evaluation' : 'results/evaluation/',
    },
    'figures': 'figures/',
}
for d in [*PATHS['output'].values(),
          PATHS['cache']['export'],
          PATHS['cache']['graph'],
          PATHS['figures']]:
    os.makedirs(d, exist_ok=True)

PARAMS = {
    'bin_size'       : 5,     # μm，10μm≈细胞直径，速度比5μm快~390x
    'qv_threshold'   : 20,
    'min_transcripts': 1,
    'k_feat'         : 15,
    'k_spatial'      : 10,
    'val_ratio'      : 0.2,
    # 'hidden_dim'     : 256,
    'hidden_dim'     : 128,
    # 'proj_dim'       : 512,
    'proj_dim'       : None,
    'dropout'        : 0.5,
    'n_epochs'       : 500,
    'lr'             : 1e-3,
    'weight_decay'   : 5e-4,
    'patience'       : 40,
    'lr_factor'      : 0.5,
    'lr_patience'    : 15,
    'min_lr'         : 1e-5,
    'max_grad_norm'  : 1.0,
    'warmup_epochs'  : 30,
    'lambda_mmd'     : 0.1,
    'lambda_ent'     : 0.01,
    'lambda_pl'      : 0.3,
    'pl_threshold'   : 0.90,
    'pl_update_freq' : 10,
    'device'         : 'cpu',
    'seed'           : 42,
}

FORCE = {
    'scrna_export': False,   # Cell 2: 重新加载 Flex 数据
    'graph'       : False,    # 首次10μm跑完后改回False（5μm旧缓存需重建）
    'train'       : False,   # Cell 4b: 重新训练模型
    'eval'        : False,   # Cell 5: 重新推断预测概率
    'seurat_cmp'  : False,   # Cell 9: 重新做 Seurat 对比
}

from utils import set_seed
set_seed(PARAMS['seed'])
from gpu_utils import list_gpus, select_gpu
list_gpus(show_processes=True)
PARAMS['device'] = select_gpu('auto', min_free_gb=8.0)

# ── 缓存文件路径 ─────────────────────────────────────────────
cache_tag = (f"kf{PARAMS['k_feat']}_ks{PARAMS['k_spatial']}"
             f"_bin{PARAMS['bin_size']}_val{PARAMS['val_ratio']}")
# 权重文件命名 tag（含模型结构参数，不同配置可共存于同一目录）
# 示例：GCN_h128_kf15_ks10_bin10.pt / GAT_h256_kf8_ks5_bin10.pt
weight_tag = (f"h{PARAMS['hidden_dim']}"
              f"_kf{PARAMS['k_feat']}_ks{PARAMS['k_spatial']}"
              f"_bin{PARAMS['bin_size']}")
GRAPH_FILE           = os.path.join(PATHS['cache']['graph'], f'graph_{cache_tag}.pt')
SCALER_FILE          = os.path.join(PATHS['cache']['graph'], f'scaler_{cache_tag}.pkl')
COORDS_FILE          = os.path.join(PATHS['cache']['graph'], f'spot_coords_{cache_tag}.npy')
RESULTS_SUMMARY_FILE = os.path.join(PATHS['output']['models'], f'gnn_results_summary_{weight_tag}.pkl')
SPOT_PROBAS_FILE     = os.path.join(PATHS['output']['predictions'], 'spot_probas_all.pkl')
EXPORT_FILES = {
    'flex_expr'         : os.path.join(PATHS['cache']['export'], 'flex_expr.npy'),
    'flex_labels'       : os.path.join(PATHS['cache']['export'], 'flex_labels.npy'),
    'cell_types'        : os.path.join(PATHS['cache']['export'], 'cell_types.json'),
    'xenium_panel_genes': os.path.join(PATHS['cache']['export'], 'xenium_panel_genes.json'),
}

print('\n=== 原始数据检查 ===')
for name, path in PATHS['raw'].items():
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path)/1e6:.1f} MB" if exists else "不存在"
    print(f'  {name:<14} {"✅" if exists else "❌"}  {size}')

print('\n=== 缓存状态 ===')
checks = [
    ('scRNA numpy 导出', all(os.path.exists(f) for f in EXPORT_FILES.values())),
    ('spot 图',          all(os.path.exists(p) for p in [GRAPH_FILE, SCALER_FILE, COORDS_FILE])),
    ('训练权重 (GCN)',   os.path.exists(os.path.join(PATHS['output']['models'], f'GCN_{weight_tag}.pt'))),
    ('训练权重 (SAGE)',  os.path.exists(os.path.join(PATHS['output']['models'], f'GraphSAGE_{weight_tag}.pt'))),
    ('训练权重 (GAT)',   os.path.exists(os.path.join(PATHS['output']['models'], f'GAT_{weight_tag}.pt'))),
    ('预测概率',         os.path.exists(SPOT_PROBAS_FILE)),
]
for name, hit in checks:
    print(f'  {"[HIT]" if hit else "[MISS]"} {name}')
print(f'\n训练设备: {PARAMS["device"]}')


┌──────────────────────────────────────────────────────────────────────┐
│ ID  型号                               空闲     已用     总计    占用 状态
├──────────────────────────────────────────────────────────────────────┤
│ [0]  Tesla V100S-PCIE-32GB         27.9GB    3.9GB   31.7GB    12%  空闲 ✓
└──────────────────────────────────────────────────────────────────────┘

  安装 pynvml+psutil 可查看占用进程
[auto] 选择 GPU 0（空闲 27.9 GB）

已选 GPU 0：Tesla V100S-PCIE-32GB
  空闲：27.9 GB / 32 GB （已用 3.9 GB）

=== 原始数据检查 ===
  transcripts    ✅  364.2 MB
  cells_csv      ✅  13.4 MB
  flex_h5        ✅  30.0 MB
  flex_annot     ✅  0.8 MB

=== 缓存状态 ===
  [HIT] scRNA numpy 导出
  [HIT] spot 图
  [MISS] 训练权重 (GCN)
  [MISS] 训练权重 (SAGE)
  [MISS] 训练权重 (GAT)
  [HIT] 预测概率

训练设备: cuda:0


In [9]:
# ── 依赖检查：pyarrow（读 parquet 必须）────────────────────────────
try:
    import pyarrow
except ImportError:
    import subprocess, sys
    print("pyarrow 未安装，正在安装...")
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "pyarrow>=14.0", "--quiet"])
    print("pyarrow 安装完成，继续...")

# ============================================================
# Cell 2: 加载 Flex scRNA-seq 数据
#
# 卵巢版改动（对比肾脏版）
# -------------------------------------------------------
# 肾脏版：rpy2 → Seurat .rds → numpy（需要先跑 R notebook）
# 卵巢版：scanpy 直接加载 .h5 + CSV 标注（无 R 依赖）
#
# Xenium panel gene 来源：从 transcripts 文件的 gene 列提取
# ============================================================
import scanpy as sc

EXPORT_OK = (all(os.path.exists(f) for f in EXPORT_FILES.values())
             and not FORCE['scrna_export'])

if EXPORT_OK:
    print('[Cache HIT] 从 numpy 缓存加载 scRNA 数据...')
    flex_expr = np.load(EXPORT_FILES['flex_expr'])
    flex_labels = np.load(EXPORT_FILES['flex_labels'])
    with open(EXPORT_FILES['cell_types']) as f:
        cell_types_list = json.load(f)
    with open(EXPORT_FILES['xenium_panel_genes']) as f:
        xenium_panel_genes = json.load(f)

else:
    print('[Cache MISS] 从 .h5 和 CSV 加载 Flex scRNA-seq...')

    # ── 1. 加载 .h5 计数矩阵 ─────────────────────────────────
    adata = sc.read_10x_h5(PATHS['raw']['flex_h5'])
    adata.var_names_make_unique()
    print(f'  原始矩阵: {adata.n_obs} 细胞 × {adata.n_vars} 基因')

    # ── 2. 加载预标注 CSV ────────────────────────────────────
    annot = pd.read_csv(PATHS['raw']['flex_annot'])
    print(f'  CSV 列名: {list(annot.columns)}')
    print(annot.head(3).to_string())

    # 自动识别 barcode 列 和 cell_type 列
    bc_col = next((c for c in annot.columns
                   if any(k in c.lower() for k in
                          ['barcode', 'cell_id', 'cellid'])), annot.columns[0])
    ct_col = next((c for c in annot.columns
                   if any(k in c.lower() for k in
                          ['cell_type', 'celltype', 'annotation',
                           'cluster_name', 'label', 'type'])), None)
    if ct_col is None:
        raise ValueError(f'找不到 cell_type 列，请手动指定。列名: {list(annot.columns)}')

    print(f'  barcode 列: {bc_col}')
    print(f'  cell_type 列: {ct_col}')

    annot = annot.set_index(bc_col)
    common_bc = adata.obs_names.intersection(annot.index)
    print(f'  有标注细胞: {len(common_bc):,} / {adata.n_obs:,}')
    if len(common_bc) == 0:
        raise ValueError('barcode 无交集！请检查 CSV barcode 格式是否与 .h5 一致')

    adata = adata[common_bc].copy()
    adata.obs['cell_type'] = annot.loc[common_bc, ct_col].values

    # ── 3. 基础 QC ────────────────────────────────────────────
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.filter_genes(adata, min_cells=3)
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
    adata = adata[adata.obs['pct_counts_mt'] < 25].copy()
    print(f'  QC 后: {adata.n_obs:,} 细胞 × {adata.n_vars:,} 基因')

    # ── 4. 过滤：只保留有明确 cell_type 的细胞 ────────────────
    valid_mask = adata.obs['cell_type'].notna() & (adata.obs['cell_type'] != '')
    adata = adata[valid_mask].copy()
    print(f'  有标注细胞: {adata.n_obs:,}')
    print('\n  细胞类型分布:')
    print(adata.obs['cell_type'].value_counts().to_string())

    # ── 5. 提取 Xenium gene panel ─────────────────────────────
    #    从 transcripts 文件的 gene 列提取，比 gene_panel.json 更可靠
    print('\n  提取 Xenium gene panel...')
    tx_path = PATHS['raw']['transcripts']
    if tx_path.endswith('.parquet'):
        tx_sample = pd.read_parquet(tx_path, columns=['feature_name'])
        xen_genes = tx_sample['feature_name'].dropna().unique().tolist()
    else:
        tx_sample = pd.read_csv(tx_path, usecols=lambda c: 'gene' in c.lower()
                                or 'feature' in c.lower(), nrows=None)
        gene_col  = [c for c in tx_sample.columns
                     if 'gene' in c.lower() or 'feature' in c.lower()][0]
        xen_genes = tx_sample[gene_col].dropna().unique().tolist()

    # 去掉 Xenium 控制探针（BLANK_ / NegCtrl / Unassigned）
    xen_genes = [g for g in xen_genes
                 if not any(p in g for p in
                            ['BLANK', 'NegCtrl', 'Unassigned', 'Negative'])]
    print(f'  Xenium 基因数: {len(xen_genes)}')

    # ── 6. 特征对齐（取共同基因）─────────────────────────────
    common_genes = sorted(set(adata.var_names) & set(xen_genes))
    print(f'  共同基因数: {len(common_genes)}')
    if len(common_genes) < 50:
        raise ValueError(f'共同基因 {len(common_genes)} 个，过少！请检查基因命名格式')

    # ── 7. 提取原始 counts 矩阵 ───────────────────────────────
    adata_sub = adata[:, common_genes]
    import scipy.sparse as sp
    if sp.issparse(adata_sub.X):
        flex_expr_raw = np.array(adata_sub.X.todense(), dtype=np.float32)
    else:
        flex_expr_raw = np.array(adata_sub.X, dtype=np.float32)

    # ── 8. 细胞类型编码 ───────────────────────────────────────
    cell_types_list = sorted(adata.obs['cell_type'].unique().tolist())
    ct2idx          = {ct: i for i, ct in enumerate(cell_types_list)}
    flex_labels     = np.array([ct2idx[ct] for ct in adata.obs['cell_type']],
                               dtype=np.int64)
    xenium_panel_genes = common_genes

    # ── 9. 保存缓存 ───────────────────────────────────────────
    np.save(EXPORT_FILES['flex_expr'],   flex_expr_raw)
    np.save(EXPORT_FILES['flex_labels'], flex_labels)
    with open(EXPORT_FILES['cell_types'], 'w') as f:
        json.dump(cell_types_list, f)
    with open(EXPORT_FILES['xenium_panel_genes'], 'w') as f:
        json.dump(xenium_panel_genes, f)

    flex_expr = flex_expr_raw
    print('\n  ✅ scRNA 缓存已保存')

# ── 汇总 ───────────────────────────────────────────────────
n_classes    = len(cell_types_list)
gene_list    = xenium_panel_genes
print(f'\nscRNA 矩阵     : {flex_expr.shape[0]:,} × {flex_expr.shape[1]}')
print(f'细胞类型数      : {n_classes}')
print(f'Xenium 基因数   : {len(gene_list)}')
print(f'\n细胞类型列表:')
for i, ct in enumerate(cell_types_list):
    n = (flex_labels == i).sum()
    print(f'  [{i:2d}] {ct:<25} n={n:,}')

[Cache HIT] 从 numpy 缓存加载 scRNA 数据...

scRNA 矩阵     : 16,569 × 453
细胞类型数      : 16
Xenium 基因数   : 453

细胞类型列表:
  [ 0] Ciliated Epithelial Cells n=165
  [ 1] Endothelial Cells         n=1,302
  [ 2] Fallopian Tube Epithelium n=295
  [ 3] Granulosa Cells           n=253
  [ 4] Inflammatory Tumor Cells  n=135
  [ 5] MT-High, Jun+/Fos+ Tumor Cells n=583
  [ 6] Macrophages               n=2,265
  [ 7] Malignant Cells Lining Cyst n=98
  [ 8] Pericytes                 n=802
  [ 9] Proliferative Tumor Cells n=1,950
  [10] Smooth Muscle Cells       n=885
  [11] Stromal Associated Fibroblasts n=1,312
  [12] T & NK Cells              n=757
  [13] Tumor Associated Fibroblasts n=2,189
  [14] Tumor Cells               n=2,255
  [15] VEGFA+ Tumor Cells        n=1,323


In [10]:
# ============================================================
# Cell 3: 加载转录本 + 构建 spot 图
# ============================================================
from utils_spot_fixed import (
    load_xenium_transcripts,
    bin_transcripts_to_spots,
    unified_normalize_spot,
    build_spot_graph,
)

GRAPH_OK = (all(os.path.exists(p)
                for p in [GRAPH_FILE, SCALER_FILE, COORDS_FILE])
            and not FORCE['graph'])

if GRAPH_OK:
    print('[Cache HIT] 加载 spot 图...')
    ckpt          = torch.load(GRAPH_FILE, weights_only=False)
    data          = ckpt['data']
    class_weights = ckpt['class_weights']
    split_info    = ckpt['split_info']
    with open(SCALER_FILE, 'rb') as f:
        fitted_scaler = pickle.load(f)
    spot_coords = np.load(COORDS_FILE)

else:
    print('[Cache MISS] 从转录本构建 spot 图...')

    df_tx = load_xenium_transcripts(
        PATHS['raw']['transcripts'],
        gene_list    = gene_list,
        qv_threshold = PARAMS['qv_threshold'],
    )
    spot_expr, spot_coords = bin_transcripts_to_spots(
        df_tx,
        gene_list       = gene_list,
        bin_size        = PARAMS['bin_size'],
        min_transcripts = PARAMS['min_transcripts'],
    )
    n_spots = spot_expr.shape[0]
    print(f'Spot 节点数: {n_spots:,}')
    if n_spots > 500_000:
        print(f'⚠️  节点数仍然较多 ({n_spots:,})，请确认 bin_size>=10μm')
    elif n_spots > 200_000:
        print(f'ℹ️  节点数适中 ({n_spots:,})，预计图构建 <5min')
    else:
        print(f'✅ 节点数合理 ({n_spots:,})')

    print('归一化...')
    scrna_norm, spot_norm, fitted_scaler = unified_normalize_spot(
        flex_expr.astype(np.float32), spot_expr)

    data, class_weights, split_info = build_spot_graph(
        scrna_norm   = scrna_norm,
        spot_norm    = spot_norm,
        spot_coords  = spot_coords,
        scrna_labels = flex_labels.astype(np.int64),
        k_feat       = PARAMS['k_feat'],
        k_spatial    = PARAMS['k_spatial'],
        val_ratio    = PARAMS['val_ratio'],
    )

    torch.save({'data': data, 'class_weights': class_weights,
                'split_info': split_info}, GRAPH_FILE)
    with open(SCALER_FILE, 'wb') as f:
        pickle.dump(fitted_scaler, f)
    np.save(COORDS_FILE, spot_coords)
    print(f'✅ 图缓存已保存 → {GRAPH_FILE}')

print(f'\n节点数    : {data.x.shape[0]:,}  '
      f'(scRNA {split_info["n_scrna"]:,} + spot {split_info["n_spots"]:,})')
print(f'边数      : {data.edge_index.shape[1]:,}')
print(f'特征维度  : {data.x.shape[1]}')
print(f'训练/验证 : {data.train_mask.sum()} / {data.val_mask.sum()}')
print('预处理完成 ✓')

[Cache HIT] 加载 spot 图...

节点数    : 1,056,772  (scRNA 16,569 + spot 1,040,203)
边数      : 34,699,484
特征维度  : 453
训练/验证 : 13255 / 3314
预处理完成 ✓


In [11]:
# ============================================================
# Cell 4a: 初始化三种 GNN 模型
# ============================================================
from models_amp import GCN_AMP, GraphSAGE_AMP, GAT_AMP
from utils import run_experiment

in_dim = data.x.shape[1]
device = torch.device(PARAMS['device'])

model_configs = {
    'GCN': GCN_AMP(
        in_channels     = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels    = n_classes,
        proj_dim        = PARAMS['proj_dim'],
        dropout         = PARAMS['dropout'],
    ),
    'GraphSAGE': GraphSAGE_AMP(
        in_channels     = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels    = n_classes,
        proj_dim        = PARAMS['proj_dim'],
        dropout         = PARAMS['dropout'],
    ),
    'GAT': GAT_AMP(
        in_channels     = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels    = n_classes,
        proj_dim        = PARAMS['proj_dim'],
        dropout         = PARAMS['dropout'],
    ),
}
print('模型参数量:')
for name, model in model_configs.items():
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  {name:<12} {n_params:,}')

模型参数量:
  GCN          77,200
  GraphSAGE    153,616
  GAT          302,768


In [12]:
# ============================================================
# Cell 4b: 训练 GNN
#
# 逻辑：
#   - 逐模型检查权重文件是否存在
#   - 只训练缺失权重的模型，跳过已有权重的模型
#   - 不同参数配置的权重文件名不同，可并存
#   - GAT 失败不影响 GCN/GraphSAGE 的加载和后续使用
#
# 权重命名格式：{ModelName}_{weight_tag}.pt
#   示例：GCN_h128_kf15_ks10_bin10.pt
#         GAT_h128_kf8_ks5_bin10.pt
# ============================================================

def _weight_file(name):
    """根据当前参数生成权重文件名（包含参数信息，不同配置可共存）"""
    return os.path.join(PATHS['output']['models'], f'{name}_{weight_tag}.pt')

# ── 逐模型检查状态 ────────────────────────────────────────────────────────
print('=== 模型权重状态检查 ===')
model_status = {}
for name in model_configs:
    wf   = _weight_file(name)
    exists = os.path.exists(wf)
    size   = f"{os.path.getsize(wf)/1e6:.1f} MB" if exists else "—"
    model_status[name] = exists
    status_icon = '✅ 已有' if exists else '⬜ 需训练'
    print(f'  {name:<12} {status_icon}  {wf}  {size}')

need_train = [n for n, ok in model_status.items() if not ok]
have_weights = [n for n, ok in model_status.items() if ok]

if FORCE['train']:
    print(f'\n[强制重训] FORCE["train"]=True → 覆盖所有已有权重')
    need_train = list(model_configs.keys())

print(f'\n需要训练: {need_train if need_train else "无（全部已有权重）"}')
print(f'直接加载: {have_weights}')

# ── 加载已有权重 ──────────────────────────────────────────────────────────
gnn_results = {}

# 尝试读取历史 summary（可能包含部分模型的历史指标）
if os.path.exists(RESULTS_SUMMARY_FILE):
    with open(RESULTS_SUMMARY_FILE, 'rb') as f:
        results_summary = pickle.load(f)
    print(f'\n已读取历史 summary: {RESULTS_SUMMARY_FILE}')
else:
    results_summary = {}

for name in have_weights:
    if FORCE['train'] and name in need_train:
        continue   # 强制重训时跳过已有权重的加载
    try:
        model = model_configs[name]
        state = torch.load(_weight_file(name), map_location='cpu', weights_only=True)
        model.load_state_dict(state)
        model.eval()
        hist_summary = results_summary.get(name, {})
        gnn_results[name] = {
            'model'        : model,
            'best_val_f1'  : hist_summary.get('best_val_f1', float('nan')),
            'best_val_acc' : hist_summary.get('best_val_acc', float('nan')),
            'best_epoch'   : hist_summary.get('best_epoch', -1),
            'history'      : hist_summary.get('history', []),
            'pred_indices' : None,   # 推断步骤补充
            'probabilities': None,
            'confidence'   : None,
        }
        print(f'  ✅ {name} 加载完成  Val F1={hist_summary.get("best_val_f1", float("nan")):.4f}')
    except Exception as e:
        print(f'  ❌ {name} 权重加载失败: {e}')
        need_train.append(name)   # 加载失败则重新训练

# ── 训练缺失权重的模型 ────────────────────────────────────────────────────
if need_train:
    print(f'\n开始训练: {need_train}')

    # 训练前先清理显存（避免已加载模型占用导致 OOM）
    for name in have_weights:
        if name in gnn_results and gnn_results[name].get('model') is not None:
            gnn_results[name]['model'] = gnn_results[name]['model'].cpu()
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'  显存清理完成，当前空闲: '
              f'{torch.cuda.mem_get_info()[0]/1024**3:.1f} GB')

    for model_name in need_train:
        if model_name not in model_configs:
            print(f'  ⚠️  {model_name} 不在 model_configs 中，跳过')
            continue

        print(f'\n{"─"*55}')
        print(f'  训练 {model_name}  (权重将保存为 {_weight_file(model_name)})')
        print(f'{"─"*55}')

        # 重新实例化模型（避免之前训练污染状态）
        from models_amp import GCN_AMP, GraphSAGE_AMP, GAT_AMP
        arch_map = {
            'GCN'       : GCN_AMP,
            'GraphSAGE' : GraphSAGE_AMP,
            'GAT'       : GAT_AMP,
        }
        ModelClass = arch_map[model_name]
        fresh_model = ModelClass(
            in_channels     = in_dim,
            hidden_channels = PARAMS['hidden_dim'],
            out_channels    = n_classes,
            proj_dim        = PARAMS['proj_dim'],
            dropout         = PARAMS['dropout'],
        )

        try:
            result = run_experiment(
                model         = fresh_model,
                data          = data,
                class_weights = class_weights,
                n_classes     = n_classes,
                device        = device,
                params        = PARAMS,
                model_name    = model_name,
                save_dir      = None,   # 不用 models.py 自带的保存，下面手动保存
            )

            # 保存带参数信息的权重文件名
            torch.save(result['model'].state_dict(), _weight_file(model_name))
            print(f'  💾 权重已保存: {_weight_file(model_name)}')

            gnn_results[model_name] = result
            results_summary[model_name] = {
                'best_val_f1'  : result['best_val_f1'],
                'best_val_acc' : result.get('best_val_acc', 0.0),
                'best_epoch'   : result.get('best_epoch', -1),
                'history'      : result.get('history', []),
                'weight_file'  : _weight_file(model_name),
                'weight_tag'   : weight_tag,
            }

            # 训练完立即保存 summary（即使后续模型失败也有记录）
            with open(RESULTS_SUMMARY_FILE, 'wb') as f:
                pickle.dump(results_summary, f)

            print(f'  ✅ {model_name} 训练完成  Val F1={result["best_val_f1"]:.4f}')

        except torch.cuda.OutOfMemoryError as oom:
            print(f'\n  ❌ {model_name} OOM: {oom}')
            print(f'  → 跳过 {model_name}，其他已完成模型不受影响')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except Exception as e:
            import traceback
            print(f'\n  ❌ {model_name} 训练异常: {type(e).__name__}: {e}')
            traceback.print_exc()
            print(f'  → 跳过 {model_name}，继续后续模型')
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # 每个模型训练后将其移回 CPU，释放显存给下一个
        if model_name in gnn_results and gnn_results[model_name].get('model'):
            gnn_results[model_name]['model'] = gnn_results[model_name]['model'].cpu()
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 将已有权重模型移回 GPU 状态的模型重新设置（只在需要时）
    for name in have_weights:
        if name in gnn_results and gnn_results[name].get('model') is not None:
            # 保持在 CPU，推断时再按需移动
            pass

# ── 最终汇总 ─────────────────────────────────────────────────────────────
print(f'\n{"="*55}')
print('模型汇总:')
for name in model_configs:
    if name in gnn_results:
        f1 = gnn_results[name].get("best_val_f1", float("nan"))
        wf = os.path.basename(_weight_file(name))
        f1_str = f"{f1:.4f}" if f1 == f1 else "N/A"
        print(f'  ✅ {name:<12} Val F1={f1_str}  ({wf})')
    else:
        print(f'  ❌ {name:<12} 未完成（权重不存在）')
print(f'{"="*55}')
print(f'\n当前 weight_tag: {weight_tag}')
print(f'权重目录: {PATHS["output"]["models"]}')
print('（不同参数配置的权重文件名不同，可安全共存）')


=== 模型权重状态检查 ===
  GCN          ⬜ 需训练  results/models/GCN_h128_kf15_ks10_bin5.pt  —
  GraphSAGE    ⬜ 需训练  results/models/GraphSAGE_h128_kf15_ks10_bin5.pt  —
  GAT          ⬜ 需训练  results/models/GAT_h128_kf15_ks10_bin5.pt  —

需要训练: ['GCN', 'GraphSAGE', 'GAT']
直接加载: []

开始训练: ['GCN', 'GraphSAGE', 'GAT']
  显存清理完成，当前空闲: 30.2 GB

───────────────────────────────────────────────────────
  训练 GCN  (权重将保存为 results/models/GCN_h128_kf15_ks10_bin5.pt)
───────────────────────────────────────────────────────

  GCN  |  GPU 空闲 30.2/32 GB


GCN:   0%|                                                                            | 0/500 [00:00<?, ?ep/s]


  ❌ GCN OOM: CUDA out of memory. Tried to allocate 17.05 GiB. GPU 
  → 跳过 GCN，其他已完成模型不受影响

───────────────────────────────────────────────────────
  训练 GraphSAGE  (权重将保存为 results/models/GraphSAGE_h128_kf15_ks10_bin5.pt)
───────────────────────────────────────────────────────

  GraphSAGE  |  GPU 空闲 27.9/32 GB


GraphSAGE:   0%|                                                                      | 0/500 [00:00<?, ?ep/s]


  ❌ GraphSAGE OOM: CUDA out of memory. Tried to allocate 58.56 GiB. GPU 
  → 跳过 GraphSAGE，其他已完成模型不受影响

───────────────────────────────────────────────────────
  训练 GAT  (权重将保存为 results/models/GAT_h128_kf15_ks10_bin5.pt)
───────────────────────────────────────────────────────

  GAT  |  GPU 空闲 27.9/32 GB


GAT:   0%|                                                                            | 0/500 [00:00<?, ?ep/s]


  ❌ GAT OOM: CUDA out of memory. Tried to allocate 68.20 GiB. GPU 
  → 跳过 GAT，其他已完成模型不受影响

模型汇总:
  ❌ GCN          未完成（权重不存在）
  ❌ GraphSAGE    未完成（权重不存在）
  ❌ GAT          未完成（权重不存在）

当前 weight_tag: h128_kf15_ks10_bin5
权重目录: results/models/
（不同参数配置的权重文件名不同，可安全共存）


In [13]:
# ============================================================
# Cell 5: 推断所有节点概率 + scRNA 验证集评估
# ============================================================
from eval import compute_metrics

val_idx  = split_info['val_idx']
val_true = flex_labels[val_idx]
n_scrna  = split_info['n_scrna']

EVAL_OK = os.path.exists(SPOT_PROBAS_FILE) and not FORCE['eval']

if EVAL_OK:
    print('[Cache HIT] 加载预测概率缓存...')
    with open(SPOT_PROBAS_FILE, 'rb') as f:
        probas_cache = pickle.load(f)
    val_preds   = probas_cache['val_preds']
    spot_probas = probas_cache['spot_probas']

else:
    print('[Cache MISS] 前向推断所有节点...')
    val_preds   = {}
    spot_probas = {}
    data_cpu    = data.cpu()

    for name, res in gnn_results.items():
        model_eval = res['model'].to('cpu').eval()
        with torch.no_grad():
            log_probs = model_eval(data_cpu)
            proba_all = torch.softmax(log_probs, dim=1).numpy()
        val_preds[name]   = proba_all[val_idx].argmax(axis=1)
        spot_probas[name] = proba_all[n_scrna:]
        print(f'  {name} 推断完成')

    with open(SPOT_PROBAS_FILE, 'wb') as f:
        pickle.dump({'val_preds': val_preds,
                     'spot_probas': spot_probas}, f)
    print(f'✅ 预测概率已保存 → {SPOT_PROBAS_FILE}')

print('\n' + '=' * 65)
print('  scRNA 验证集评估（Ground Truth = Flex 预标注细胞类型）')
print('=' * 65)
print(f'{"Method":<14} {"Acc":>7} {"F1-Mac":>8} {"F1-Wei":>8} {"Kappa":>8}')
print('-' * 65)
for method, y_pred in val_preds.items():
    m = compute_metrics(val_true, y_pred, n_classes=n_classes)
    print(f'{method:<14} {m["accuracy"]:>7.4f} {m["f1_macro"]:>8.4f} '
          f'{m["f1_weighted"]:>8.4f} {m["kappa"]:>8.4f}')
print('=' * 65)

[Cache HIT] 加载预测概率缓存...

  scRNA 验证集评估（Ground Truth = Flex 预标注细胞类型）
Method             Acc   F1-Mac   F1-Wei    Kappa
-----------------------------------------------------------------
GCN             0.5625   0.4414   0.5811   0.5219
GraphSAGE       0.8105   0.7425   0.8146   0.7910


In [14]:
# ============================================================
# Cell 6: 保存 spot 级别预测结果（CSV）
# ============================================================
best_name = max(gnn_results, key=lambda n: gnn_results[n]['best_val_f1'])
print(f'最优模型: {best_name}  Val F1={gnn_results[best_name]["best_val_f1"]:.4f}')

best_proba      = spot_probas[best_name]
best_pred_idx   = best_proba.argmax(axis=1)
best_pred_label = [cell_types_list[i] for i in best_pred_idx]
best_confidence = best_proba.max(axis=1)

spot_df = pd.DataFrame({
    'x'         : spot_coords[:, 0],
    'y'         : spot_coords[:, 1],
    'pred_idx'  : best_pred_idx,
    'pred_label': best_pred_label,
    'confidence': best_confidence,
})
out_csv = f'{PATHS["output"]["predictions"]}/spot_predictions_{best_name}.csv'
spot_df.to_csv(out_csv, index=False)
print(f'Spot 预测已保存: {out_csv}  ({len(spot_df):,} spots)')

for name, proba in spot_probas.items():
    df_all = pd.DataFrame(proba,
        columns=[f'prob_{c}' for c in cell_types_list])
    df_all['x']          = spot_coords[:, 0]
    df_all['y']          = spot_coords[:, 1]
    df_all['pred_label'] = [cell_types_list[i]
                            for i in proba.argmax(axis=1)]
    df_all.to_csv(
        f'{PATHS["output"]["predictions"]}/spot_predictions_{name}_full.csv',
        index=False)
print('所有模型 spot 预测已保存 ✓')

ValueError: max() arg is an empty sequence

In [ ]:
# ============================================================
# Cell 7: 空间可视化
# 修复：PATHS['figures'] 替代原错误的 PATHS['output']['plots']
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, len(gnn_results),
                         figsize=(7*len(gnn_results), 7))
if len(gnn_results) == 1:
    axes = [axes]

cmap   = plt.cm.get_cmap('tab20', n_classes)
colors = {i: cmap(i) for i in range(n_classes)}

for ax, (model_name, proba) in zip(axes, spot_probas.items()):
    pred_idx = proba.argmax(axis=1)
    c_vals   = [colors[i] for i in pred_idx]
    ax.scatter(spot_coords[:, 0], spot_coords[:, 1],
               c=c_vals, s=0.3, alpha=0.5, rasterized=True)
    ax.set_aspect('equal')
    ax.set_title(
        f'{model_name}\nVal F1={gnn_results[model_name]["best_val_f1"]:.3f}',
        fontsize=12)
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('y (μm)')

patches = [mpatches.Patch(color=colors[i], label=cell_types_list[i])
           for i in range(n_classes)]
fig.legend(handles=patches, bbox_to_anchor=(1.01, 0.9),
           loc='upper left', fontsize=8, framealpha=0.8)
plt.suptitle(
    f'GNN Spot-level Cell Type Predictions\n'
    f'Ovarian Cancer Xenium | bin={PARAMS["bin_size"]}μm | '
    f'No Cell Segmentation Required',
    fontsize=13, y=1.02)
plt.tight_layout()
# 修复：使用 PATHS['figures'] 而非 PATHS['output']['plots']
out_fig = f'{PATHS["figures"]}/spot_spatial_all_models.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'空间图已保存: {out_fig}')

In [ ]:
# ============================================================
# Cell 8: Moran's I 空间自相关
# ============================================================
from topact import TopACT

best_pred_int = spot_probas[best_name].argmax(axis=1)

print(f"计算全局 Moran's I（{best_name}）...")
moran_global = TopACT.morans_i(
    labels      = best_pred_int,
    coords      = spot_coords,
    n_neighbors = 10,
)
print(f"全局 Moran's I = {moran_global['I']:.4f}"
      f" (z={moran_global['z_score']:.2f}, p={moran_global['p_value']:.2e})")

print(f"\n各细胞类型 Moran's I:")
per_class = TopACT.per_class_morans_i(
    labels      = best_pred_int,
    coords      = spot_coords,
    cell_types  = cell_types_list,
    n_neighbors = 10,
)
for ct, vals in sorted(per_class.items(), key=lambda x: -x[1]['I']):
    print(f'  {ct:<25} I={vals["I"]:>7.4f}  p={vals["p_value"]:.2e}')

In [ ]:
# ============================================================
# Cell 9: Seurat 标签转移 vs GNN 对比分析
#
# Seurat TransferData：细胞级别预测（依赖细胞分割）
# 本方法（GNN）：  spot 级别预测（不依赖细胞分割）
#
# 对比维度
# --------
# 1. 空间分布对比图（4-panel：GCN/SAGE/GAT/Seurat）
# 2. 细胞类型比例对比（堆叠条形图）
# 3. GNN spot 预测聚合到细胞质心 → 与 Seurat 直接比较一致性
#
# 注意：Seurat 结果没有验证集 F1（Xenium 无 ground truth），
# 无法直接比较数值指标，只能做空间分布合理性的定性对比。
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from utils_spot_fixed import aggregate_spot_to_cell

SEURAT_LT_FILE = 'data/cache/xenium/seurat_label_transfer_ovarian.csv'

if not os.path.exists(SEURAT_LT_FILE):
    print(f'⚠️  Seurat 标签转移结果不存在：{SEURAT_LT_FILE}')
    print('   请先运行 labelTransfer_ovarian.ipynb 的 Cell 4b')
    print('   跳过 Seurat 对比，只展示 GNN 结果')
    has_seurat = False
else:
    seurat_df  = pd.read_csv(SEURAT_LT_FILE)
    has_seurat = True
    print(f'Seurat 预测加载完成: {len(seurat_df):,} 个细胞')
    print(f'  列名: {list(seurat_df.columns)}')
    print(f'  预测类型数: {seurat_df["predicted_id"].nunique()}')
    print(f'  平均置信度: {seurat_df["prediction_score"].mean():.3f}')

# ── 1. 空间分布对比图（4-panel）─────────────────────────────
print('\n绘制空间对比图...')

# 统一颜色映射（所有方法共用同一套颜色）
all_types = sorted(set(cell_types_list))
if has_seurat:
    seurat_types = seurat_df['predicted_id'].dropna().unique().tolist()
    all_types    = sorted(set(all_types) | set(seurat_types))
cmap   = plt.cm.get_cmap('tab20', len(all_types))
type2color = {t: cmap(i) for i, t in enumerate(all_types)}

n_panels = len(gnn_results) + (1 if has_seurat else 0)
fig, axes = plt.subplots(1, n_panels,
                         figsize=(7 * n_panels, 7))
if n_panels == 1:
    axes = [axes]

# GNN 方法（spot 级别）
for ax, (model_name, proba) in zip(axes, spot_probas.items()):
    pred_labels = [cell_types_list[i] for i in proba.argmax(axis=1)]
    c_vals      = [type2color[l] for l in pred_labels]
    ax.scatter(spot_coords[:, 0], spot_coords[:, 1],
               c=c_vals, s=0.3, alpha=0.6, rasterized=True)
    ax.set_aspect('equal')
    ax.set_title(
        f'{model_name} (GNN Spot-level)\n'
        f'Val F1={gnn_results[model_name]["best_val_f1"]:.3f}',
        fontsize=11)
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('y (μm)')
    ax.tick_params(labelsize=8)

# Seurat（细胞级别）
if has_seurat:
    ax = axes[-1]
    valid = seurat_df['predicted_id'].notna()
    c_vals = [type2color.get(t, (0.5, 0.5, 0.5, 1))
               for t in seurat_df.loc[valid, 'predicted_id']]
    ax.scatter(
        seurat_df.loc[valid, 'x'],
        seurat_df.loc[valid, 'y'],
        c=c_vals, s=1.0, alpha=0.6, rasterized=True
    )
    mean_score = seurat_df['prediction_score'].mean()
    ax.set_aspect('equal')
    ax.set_title(
        f'Seurat TransferData (Cell-level)\n'
        f'Mean Score={mean_score:.3f}  ★ Requires Cell Segmentation',
        fontsize=11)
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('y (μm)')
    ax.tick_params(labelsize=8)

# 共享图例
patches = [mpatches.Patch(color=type2color[t], label=t) for t in all_types]
fig.legend(handles=patches, bbox_to_anchor=(1.01, 0.95),
           loc='upper left', fontsize=8, framealpha=0.8,
           title='Cell Type')
fig.suptitle(
    'Spatial Cell Type Prediction Comparison\n'
    'Ovarian Cancer Xenium FFPE — GNN (no segmentation) vs Seurat (cell segmentation)',
    fontsize=12, y=1.02)
plt.tight_layout()
out_compare = f'{PATHS["figures"]}/spatial_comparison_gnn_vs_seurat.png'
plt.savefig(out_compare, dpi=150, bbox_inches='tight')
plt.show()
print(f'空间对比图已保存: {out_compare}')

# ── 2. 细胞类型比例对比（堆叠条形图）────────────────────────
print('\n绘制细胞比例对比...')

def get_proportions(labels_or_series, type_list):
    if hasattr(labels_or_series, 'value_counts'):
        counts = labels_or_series.value_counts()
        total  = counts.sum()
        return {t: counts.get(t, 0) / total for t in type_list}
    else:
        labels = np.asarray(labels_or_series)
        total  = len(labels)
        return {t: (labels == i).sum() / total
                for i, t in enumerate(cell_types_list) if t in type_list}

sources = {}
# scRNA 参考分布
n_per_type = np.bincount(flex_labels, minlength=n_classes)
total_sc   = n_per_type.sum()
sources['scRNA (ref)'] = {
    t: n_per_type[i] / total_sc
    for i, t in enumerate(cell_types_list)
}
# GNN 方法
for model_name, proba in spot_probas.items():
    pred_idx = proba.argmax(axis=1)
    n_per    = np.bincount(pred_idx, minlength=n_classes)
    sources[model_name] = {
        t: n_per[i] / n_per.sum()
        for i, t in enumerate(cell_types_list)
    }
# Seurat
if has_seurat:
    vc    = seurat_df['predicted_id'].value_counts()
    total = vc.sum()
    sources['Seurat'] = {
        t: vc.get(t, 0) / total for t in all_types
    }

prop_df = pd.DataFrame(sources, index=all_types).fillna(0).T
colors  = [type2color[t] for t in all_types]

fig, ax = plt.subplots(figsize=(max(10, len(sources) * 2), 5))
prop_df.plot(kind='bar', stacked=True, color=colors,
             ax=ax, width=0.7, legend=False)
ax.set_ylabel('Proportion')
ax.set_ylim(0, 1)
ax.set_title('Cell Type Proportion: scRNA ref vs GNN vs Seurat')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
handles = [mpatches.Patch(color=colors[i], label=t)
           for i, t in enumerate(all_types)]
ax.legend(handles=handles, bbox_to_anchor=(1.02, 1),
          loc='upper left', fontsize=8, frameon=False)
fig.tight_layout()
out_prop = f'{PATHS["figures"]}/proportion_comparison.png'
fig.savefig(out_prop, dpi=150, bbox_inches='tight')
plt.show()
print(f'比例对比图已保存: {out_prop}')

# ── 3. GNN spot → 细胞级别聚合，与 Seurat 一致性分析 ────────
if has_seurat:
    print('\n计算 GNN spot 预测 → 细胞级别聚合...')
    cell_coords_arr = seurat_df[['x', 'y']].values.astype(np.float32)

    best_proba_arr = spot_probas[best_name]
    cell_pred_idx, cell_confidence = aggregate_spot_to_cell(
        spot_proba  = best_proba_arr,
        spot_coords = spot_coords,
        cell_coords = cell_coords_arr,
        n_classes   = n_classes,
        radius_um   = 10.0,
    )
    gnn_cell_labels  = [cell_types_list[i] for i in cell_pred_idx]
    seurat_labels_aligned = seurat_df['predicted_id'].fillna('Unknown').values

    # 一致性（agreement）
    agree = np.array(gnn_cell_labels) == seurat_labels_aligned
    agreement_rate = agree.mean()
    print(f'\nGNN ({best_name}) vs Seurat 细胞级别一致性: {agreement_rate:.1%}')
    print('（注：两者均无 ground truth，一致率高说明两种方法给出相近答案，不代表哪个正确）')

    # 按细胞类型分组看一致性
    print('\n各细胞类型一致性：')
    agree_df = pd.DataFrame({
        'seurat'    : seurat_labels_aligned,
        'gnn'       : gnn_cell_labels,
        'agree'     : agree,
        'confidence': cell_confidence,
    })
    per_type = agree_df.groupby('seurat')['agree'].agg(['mean', 'count'])
    per_type.columns = ['agreement', 'n_cells']
    per_type = per_type.sort_values('agreement', ascending=False)
    print(per_type.to_string())

    # 保存对比 CSV
    compare_csv = f'{PATHS["output"]["evaluation"]}/gnn_vs_seurat_cell_comparison.csv'
    agree_df.to_csv(compare_csv, index=False)
    print(f'\n细胞级对比已保存: {compare_csv}')

print('\n✅ Seurat 对比分析完成')